<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/Stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [1]:
import pandas as pd
import json
import os
import glob

Set the Path to the Data location (imported form SharpHound)

# Set the Path
to the Data location (imported from SharpHound) and Read json files in folder and create a dictionary with the entire data.





In [2]:
#@title Download lab files
# from IPython.display import clear_output
# if not os.path.exists("/content/mylab"):
#   print("Downloading lab files...")
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_computers.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_containers.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_domains.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_gpos.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_groups.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_ous.json
#  !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab/20250316212118_users.json
#   !wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/mylab.zip
#   !unzip mylab.zip -d mylab
#   clear_output()
#   print("Done")
!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
#!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure10k.json
# else:
#   print("Lab files already downloaded")
!mkdir = "/content/adsync"
!mv secure.json /content/adsync
#!mv secure10k.json /content/adsync

path = "/content/adsync"
data = {}
json_files = glob.glob(os.path.join(path, "*.json"))
for file in json_files:
    filename = os.path.basename(file).replace(".json", "")
    suf = filename.split("_")[-1]

    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)

    if "data" in content:
        if suf in data:
            data[f"{suf}"].extend(content["data"])
            #print(f" The key 'data' was found in {f}.")
        else:
            data[f"{suf}"] = content["data"]
            #print(f" The key 'data' was found in {f}.")
    else:
        print(f"Warning: The key 'data' was not found in {f}.")

--2025-07-13 12:35:45--  https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1314149 (1.3M) [text/plain]
Saving to: ‘secure.json’

secure.json         100%[===================>]   1.25M  --.-KB/s    in 0.08s   

2025-07-13 12:35:46 (15.2 MB/s) - ‘secure.json’ saved [1314149/1314149]



Show the "content"

In [3]:
content


{'data': [{'id': '0',
   'labels': ['Base', 'Domain'],
   'properties': {'domain': 'MARK.LOCAL',
    'name': 'MARK.LOCAL',
    'highvalue': True,
    'objectid': 'S-1-5-21-883232822-274137685-4173207997',
    'distinguishedname': 'DC=MARK,DC=LOCAL',
    'functionallevel': '2012',
    'owned': False},
   'type': 'node'},
  {'id': '1',
   'labels': ['Base', 'OU'],
   'properties': {'domain': 'MARK.LOCAL',
    'name': 'Admin@MARK.LOCAL',
    'objectid': 'S-1-5-21-883232822-274137685-4173207997-cdca22b1-d164-4ec1-8233-45c0ea9044ac',
    'distinguishedname': 'OU=ADMIN,DC=MARK,DC=LOCAL',
    'description': None,
    'highvalue': False,
    'blocksInheritance': False,
    'owned': False},
   'type': 'node'},
  {'id': '2',
   'labels': ['Base', 'OU'],
   'properties': {'domain': 'MARK.LOCAL',
    'name': 'Tier 1 Servers@MARK.LOCAL',
    'objectid': 'S-1-5-21-883232822-274137685-4173207997-f6d49f51-1ea3-4d9c-93a1-0b591b59c7e9',
    'distinguishedname': 'OU=TIER 1 SERVERS,DC=MARK,DC=LOCAL',
    

In [4]:
# prompt: from data['secure'] get the type column

# try:
#   secure_types = [item['type'] for item in data[suf]]
#   print(secure_types)
# except KeyError:
#   print("The key 'secure' was not found in the data dictionary.")
# except TypeError:
#   print("The value for 'secure' is not a list of dictionaries.")

In [5]:
# prompt: if the type column = "node" save that record into a pandas dataframe called nodes, if the type = "relationship' then save to a pandas dataframe called edges

import pandas as pd
df = pd.DataFrame(data[suf])
nodes = df[df['type'] == 'node'].copy()
edges = df[df['type'] == 'relationship'].copy()

In [6]:
# prompt: convert id in nodes to int

nodes['id'] = nodes['id'].astype(int)
#nodes.head()

In [7]:
nodes['id'].dtype

dtype('int64')

In [8]:
# prompt: show all rows

import pandas as pd
pd.set_option('display.max_rows', None)



In [9]:
# prompt: for every record in "nodes" create a l1, l2 and l3 column for the values in "nodes.labels" eg. for record 1 l1=Base, l2=OU and l3 would be null

import pandas as pd
def extract_labels(labels):
    # Ensure labels is a list
    if not isinstance(labels, list):
        return [None, None, None]

    l1 = labels[0] if len(labels) > 0 else None
    l2 = labels[1] if len(labels) > 1 else None
    l3 = labels[2] if len(labels) > 2 else None
    return [l1, l2, l3]

# Apply the function to create the new columns
nodes[['l1', 'l2', 'l3']] = nodes['labels'].apply(lambda x: pd.Series(extract_labels(x)))

#nodes.head()

In [10]:
# prompt: show me records where l3 ne "None"

nodes[nodes['l3'].notna()]

,id,labels,properties,type,start,end,label,l1,l2,l3
71,71,"[Base, Group, User]","{'domain': 'MARK.LOCAL', 'name': 'DMAPLE00091@...",node,NaN,NaN,NaN,Base,Group,User
83,83,"[Base, Group, User]","{'domain': 'MARK.LOCAL', 'name': 'MHONNETTE000...",node,NaN,NaN,NaN,Base,Group,User
396,396,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
425,425,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
428,428,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised
450,450,"[Base, User, Compromised]","{'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...",node,NaN,NaN,NaN,Base,User,Compromised


In [11]:
# prompt: remove the following columns from nodes:
# start, end, label

nodes = nodes.drop(columns=['start', 'end', 'label', 'labels'])
#nodes.head()

In [12]:
#edges.tail()

In [13]:
# prompt: remove the "labels" column from edges

edges = edges.drop(columns=['labels'])
#edges.head()

Expand "Properties" column in new columns.

Convert to Dict

In [14]:
pd.set_option("display.max_rows", None, "display.max_columns", None)

#nodes

In [15]:
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
edges.head()
edges['id'].dtype

dtype('int64')

In [16]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
nodes_json = nodes.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('nodes.json', 'w') as f:
  f.write(nodes_json)


In [17]:
# prompt: save the df_compuers as json to content

import json

# Assuming df_computers is already defined as in your provided code.

# Convert the DataFrame to a JSON string.
edges_json = edges.to_json(orient='records')

# Save the JSON string to a file named 'content.json'.
with open('edges.json', 'w') as f:
  f.write(edges_json)

In [18]:
# prompt: create a new column in nodes called weight

nodes['weight'] = 1

In [19]:
# prompt: create a list of all id from nodes where "highvalue" = true

# Before accessing 'Properties', check if the column exists in the DataFrame.
if 'properties' in nodes.columns:
    # Use .loc to avoid SettingWithCopyWarning and access the nested 'highvalue' key
    # We use a lambda function with .apply to safely access nested dictionary keys.
    high_value_node_ids = nodes[nodes['properties'].apply(lambda x: x.get('highvalue', False) == True)]['id'].tolist()
    print(high_value_node_ids)
else:
    print("Error: The 'Properties' column was not found in the nodes DataFrame.")

[0, 36, 38, 40, 59, 60, 65, 66, 72, 75, 90, 91, 92, 93, 94, 95, 96, 97, 98, 288, 289, 291, 293, 294, 295, 296, 297, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 324, 491, 501, 507, 508, 509, 511, 512, 514, 521, 523, 527, 530, 531, 532, 533, 534, 539, 544, 548, 550, 553, 557, 561, 562, 563, 564, 565, 567, 573, 575, 577, 585, 590, 597, 599, 600, 603, 607, 609, 610, 620, 623, 624, 625, 626, 631, 632, 637, 638, 639, 640, 645, 648, 649, 650, 651, 653, 654, 655, 656, 660, 661, 663, 665, 666, 668, 672, 673, 676, 677, 681, 684, 692, 693, 694]


In [20]:
len(high_value_node_ids)


124

In [21]:
# prompt: if node[id] = value in high_value_nodes_ids assign a weight of 20

for value in high_value_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 20


In [22]:
# prompt: show me nodes records where weight = 20

nodes[nodes['weight'] == 20]

,id,properties,type,l1,l2,l3,weight
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL',...",node,Base,Domain,None,20
36,36,"{'domain': 'MARK.LOCAL', 'name': 'ADMINISTRATO...",node,Base,Group,None,20
38,38,"{'domain': 'MARK.LOCAL', 'name': 'PRINT OPERAT...",node,Base,Group,None,20
40,40,"{'domain': 'MARK.LOCAL', 'name': 'BACKUP OPERA...",node,Base,Group,None,20
59,59,"{'domain': 'MARK.LOCAL', 'name': 'SERVER OPERA...",node,Base,Group,None,20
60,60,"{'domain': 'MARK.LOCAL', 'name': 'ACCOUNT OPER...",node,Base,Group,None,20
65,65,"{'domain': 'MARK.LOCAL', 'name': 'ENTERPRISE A...",node,Base,Group,None,20
66,66,"{'domain': 'MARK.LOCAL', 'name': 'DOMAIN ADMIN...",node,Base,Group,None,20
72,72,"{'domain': 'MARK.LOCAL', 'name': 'DOMAIN CONTR...",node,Base,Group,None,20
75,75,"{'domain': 'MARK.LOCAL', 'name': 'GROUP POLICY...",node,Base,Group,None,20


In [23]:
len(nodes[nodes['weight'] == 20])

124

In [24]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
pd.set_option('display.max_colwidth', None)

#nodes.tail(50)

In [25]:
# prompt: print all nodes where weight==20

#print(nodes[nodes['weight'] == 20])

In [26]:
len(nodes)

1297

In [27]:
# prompt: show me records where nodes['properties']['distinguishedname']  does not contain "T2", "T1" or "T0"

# Filter nodes where 'distinguishedname' in 'properties' does not contain "T2", "T1", or "T0"
filtered_nodes = nodes[
    nodes['properties'].apply(
        lambda x: 'distinguishedname' in x and
                  'T2' not in x['distinguishedname'] and
                  'T1' not in x['distinguishedname'] and
                  'T0' not in x['distinguishedname'] and
                  'Tier 0' not in x['distinguishedname'] and
                  'Tier 1' not in x['distinguishedname'] and
                  'Tier 2' not in x['distinguishedname'] and
                  'TIER 0' not in x['distinguishedname'] and
                  'TIER 1' not in x['distinguishedname'] and
                  'TIER 2' not in x['distinguishedname']
    )
]

filtered_nodes


,id,properties,type,l1,l2,l3,weight
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL', 'highvalue': True, 'objectid': 'S-1-5-21-883232822-274137685-4173207997', 'distinguishedname': 'DC=MARK,DC=LOCAL', 'functionallevel': '2012', 'owned': False}",node,Base,Domain,None,20
1,1,"{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cdca22b1-d164-4ec1-8233-45c0ea9044ac', 'distinguishedname': 'OU=ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
19,19,"{'domain': 'MARK.LOCAL', 'name': 'Application@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-c8fbc840-cc56-4337-aa2c-31984398cf7f', 'distinguishedname': 'OU=APPLICATION,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
20,20,"{'domain': 'MARK.LOCAL', 'name': 'Print@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-70355ec3-a132-4ec7-b4c7-f7cdfb011b59', 'distinguishedname': 'OU=PRINT,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
21,21,"{'domain': 'MARK.LOCAL', 'name': 'Database@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-bee8279d-bcc2-427c-827c-510159969430', 'distinguishedname': 'OU=DATABASE,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
36,36,"{'domain': 'MARK.LOCAL', 'name': 'ADMINISTRATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-544', 'highvalue': True, 'distinguishedname': 'CN=Administrators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Administrators have complete and unrestricted access to the computer/domain', 'admincount': True, 'owned': False}",node,Base,Group,None,20
37,37,"{'domain': 'MARK.LOCAL', 'name': 'REMOTE DESKTOP USERS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-555', 'highvalue': False, 'distinguishedname': 'CN=Remote Desktop Users,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members in this group are granted the right to logon remotely', 'admincount': False, 'owned': False}",node,Base,Group,None,1
38,38,"{'domain': 'MARK.LOCAL', 'name': 'PRINT OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-550', 'highvalue': True, 'distinguishedname': 'CN=Print Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer printers installed on domain controllers', 'admincount': True, 'owned': False}",node,Base,Group,None,20
39,39,"{'domain': 'MARK.LOCAL', 'name': 'IIS_IUSRS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-568', 'highvalue': False, 'distinguishedname': 'CN=IIS_IUSRS,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Built-in group used by Internet Information Services.', 'admincount': False, 'owned': False}",node,Base,Group,None,1
40,40,"{'domain': 'MARK.LOCAL', 'name': 'BACKUP OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-551', 'highvalue': True, 'distinguishedname': 'CN=Backup Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Backup Operators can override security restrictions for the sole purpose of backing up or restoring files', 'admincount': True, 'owned': False}",node,Base,Group,None,20


In [28]:
len(filtered_nodes)

68

In [29]:
# prompt: show all column contents for nodes do not truncate columns

import pandas as pd
pd.set_option('display.max_colwidth', None)
#nodes

In [30]:
# prompt: compile a list of all id's from nodes where distinguishedname contains 'Tier 0' or T0. distinguishedname is a key within the properties column

# tier_0_t0_ids = []
# for index, row in nodes.iterrows():
#     properties = row.get('properties')
#     if properties and isinstance(properties, dict):
#         distinguishedname = properties.get('distinguishedname')
#         if distinguishedname and ('Tier 0' in distinguishedname or 'T0' in distinguishedname or 'TIER 0' in distinguishedname):
#             tier_0_t0_ids.append(row['id'])

# print("tier 0: ", tier_0_t0_ids)
# len_tier0=len(tier_0_t0_ids)
# print(len_tier0)

# tier_1_t0_ids = []
# for index, row in nodes.iterrows():
#     properties = row.get('properties')
#     if properties and isinstance(properties, dict):
#         distinguishedname = properties.get('distinguishedname')
#         if distinguishedname and ('Tier 1' in distinguishedname or 'T1' in distinguishedname or 'TIER 1' in distinguishedname):
#             tier_1_t0_ids.append(row['id'])

# print("tier 1: ",tier_1_t0_ids)
# len_tier1=len(tier_1_t0_ids)
# print(len_tier1)

# tier_2_t0_ids = []
# for index, row in nodes.iterrows():
#     properties = row.get('properties')
#     if properties and isinstance(properties, dict):
#         distinguishedname = properties.get('distinguishedname')
#         if distinguishedname and ('Tier 2' in distinguishedname or 'T2' in distinguishedname or 'TIER 2' in distinguishedname):
#             tier_2_t0_ids.append(row['id'])

# print("tier 2: ",tier_2_t0_ids)
# len_tier2= len(tier_2_t0_ids)
# print(len_tier2)
# total = len_tier0 + len_tier1 + len_tier2
# print("total: ",total)


In [31]:
# prompt: if tier_0_t0_ids isnot high_value_node_ids then append nodeid in a new dataframe called total_tier0

# total_tier0 = []
# for nodeid in tier_0_t0_ids:
#   if nodeid not in high_value_node_ids:
#     total_tier0.append(nodeid)

# print("Nodes in tier_0_t0_ids that are not in high_value_node_ids:", total_tier0)

In [32]:
# prompt: total_tier0 = total_tier0 and appended high_value_node_ids

# total_tier0.extend(high_value_node_ids)
# print("total_tier0:", total_tier0)


In [33]:
#len(total_tier0)

In [34]:
len(nodes)

1297

In [35]:
# prompt: how do i rewrite the following code to check for "TIER 0", "Tier 0" and "T0":
# tier0_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 0' in x.get('distinguishedname', ''))]['id'].tolist()

# tier0_node_ids = nodes[
#     nodes['properties'].apply(
#         lambda x: any(
#             term in x.get('distinguishedname', '')
#             for term in ['TIER 0', 'Tier 0', 'T0']
#         )
#     )
# ]['id'].tolist()


In [36]:
#tier2_node_ids

In [37]:
# 1/0
# tier0_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 0' in x.get('distinguishedname', ''))]['id'].tolist()
# tier0_node_ids

tier0_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 0', 'Tier 0', 'T0']
        )
    )
]['id'].tolist()


tier1_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 1', 'Tier 1', 'T1']
        )
    )
]['id'].tolist()

tier2_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 2', 'Tier 2', 'T2']
        )
    )
]['id'].tolist()

print(f"tier 0 ",len(tier0_node_ids))
# tier1_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 1' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 1 ",len(tier1_node_ids))

# tier2_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 2' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 2 ",len(tier2_node_ids))

tier 0  42
tier 1  58
tier 2  949


In [38]:
len(nodes)

1297

In [39]:
# T0="T0"
# T1="T1"
# T2="T2"


In [40]:
# # prompt: produce a list of id's from nodes when nodes[properties][name] contains T0

# ids_with_T0 = nodes[nodes['properties'].apply(lambda x: T0 in x.get('name', ''))]['id'].tolist()
# #ids_with_T0
# ids_with_T1 = nodes[nodes['properties'].apply(lambda x: T1 in x.get('name', ''))]['id'].tolist()
# #ids_with_T1
# ids_with_T2 = nodes[nodes['properties'].apply(lambda x: T2 in x.get('name', ''))]['id'].tolist()
# #ids_with_T2

In [41]:
# print(ids_with_T0)
# print(ids_with_T1)
# print(ids_with_T2)

# print(len(ids_with_T0))
# print(len(ids_with_T1))
# print(len(ids_with_T2))


In [42]:

for value in tier0_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 20

for value in tier1_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 10

for value in tier2_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 5

nodes.head(50)

,id,properties,type,l1,l2,l3,weight
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL', 'highvalue': True, 'objectid': 'S-1-5-21-883232822-274137685-4173207997', 'distinguishedname': 'DC=MARK,DC=LOCAL', 'functionallevel': '2012', 'owned': False}",node,Base,Domain,None,20
1,1,"{'domain': 'MARK.LOCAL', 'name': 'Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cdca22b1-d164-4ec1-8233-45c0ea9044ac', 'distinguishedname': 'OU=ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
2,2,"{'domain': 'MARK.LOCAL', 'name': 'Tier 1 Servers@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f6d49f51-1ea3-4d9c-93a1-0b591b59c7e9', 'distinguishedname': 'OU=TIER 1 SERVERS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,10
3,3,"{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5de20ac8-e7ff-4f05-b597-268af392400e', 'distinguishedname': 'OU=TIER 2,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,5
4,4,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f249f685-d5e0-4dad-b5b5-7f723c41ca1e', 'distinguishedname': 'OU=T0 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
5,5,"{'domain': 'MARK.LOCAL', 'name': 'T1 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-8355af01-1b4e-48c8-8977-a53870b2770b', 'distinguishedname': 'OU=T1 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,10
6,6,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e4602f9d-12bf-4f9e-b667-f6ccd27fae90', 'distinguishedname': 'OU=T2 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,5
7,7,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-c0fb853d-9b96-449b-b25c-4d20a1c02e5f', 'distinguishedname': 'OU=T0 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
8,8,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-b9fd1aed-b3e6-4d53-9242-b85ff389822f', 'distinguishedname': 'OU=T0 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
9,9,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cd13b9d6-76ca-4b5c-82bd-2d86bef8a375', 'distinguishedname': 'OU=T0 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20


In [43]:
# prompt: get a unique list of label from edges and save it to list called edges_labels

edges_labels = edges['label'].unique().tolist()
edges_labels

['Contains',
 'GpLink',
 'GenericAll',
 'MemberOf',
 'GenericWrite',
 'WriteDacl',
 'Owns',
 'WriteOwner',
 'GetChanges',
 'GetChangesAll',
 'AllExtendedRights',
 'AdminTo',
 'HasSession',
 'ReadLAPSPassword',
 'CanRDP',
 'ExecuteDCOM',
 'AddSelf',
 'ForceChangePassword',
 'AddMember']

In [44]:
# Apply the weights based on the 'label' column

def assign_weight(label):
    if label == "GenericAll":
        return 9
    elif label == "Owns":
        return 10
    elif label == "GenericWrite":
        return 2
    elif label == "AllExtendedRights":
        return 6
    elif label == "CanRDP":
        return 2
    elif label == "Contains":
        return 2
    elif label == "DCSync":
        return 8
    elif label == "WriteDacl":
        return 5
    elif label == "WriteOwner":
        return 7
    elif label == "AddKeyCredentialLink":
        return 6
    elif label == "AdminTo":
        return 8
    elif label == "MemberOf":
        return 1
    elif label == "CanPSRemote":
        return 2
    elif label == "ExecuteDCOM":
        return 2
    elif label == "GPLink":
        return 3
    elif label == "HasSession":
        return 9
    elif label == "ReadLAPSPassword":
        return 2
    elif label == "GetChanges":
        return 3
    elif label == "GetChangesAll":
        return 3
    elif label == "AddSelf":
        return 3
    elif label == "ForceChangePassword":
        return 3
    elif label == "AddMember":
        return 5
    elif label == "tier0":
        return 20
    else:
        return 1 # Default weight for other labels

edges['weight'] = edges['label'].apply(assign_weight)

# Display the updated edges_all DataFrame with the 'weight' column
print("\nEdges_all DataFrame with weight column:")
#edges.head(50)
#clear_output()


Edges_all DataFrame with weight column:


In [45]:
# prompt: from edges['start'] get 'id' and from edges['end] get 'id'
# replace start with edges['start'][id'] and  end with edges['end']['id']

edges['start'] = edges['start'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
edges['end'] = edges['end'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
#edges.head(30)

In [46]:
len(edges)

3974

In [47]:
print("\nEdges_all DataFrame with weight column:")
#edges.tail(50)


Edges_all DataFrame with weight column:


In [48]:
# prompt: create a new dataframe called adj with the values of start['id'] and end['id'] from each record

# adj = edges[['start', 'end']].copy()
# adj['start'] = adj['start'].apply(lambda x: x['id']).astype(int)
# adj['end'] = adj['end'].apply(lambda x: x['id']).astype(int)
# adj.head()

In [49]:
# len(edges)
# edges.head(20)

In [50]:
# prompt: from edges, convert start[id] into an integer

# edges['start'] = edges['start'].apply(lambda x: x['id']).astype(int)

In [51]:
# edges['end'] = edges['end'].apply(lambda x: x['id']).astype(int)
# edges['start'] = edges['start'].apply(lambda x: x['id']).astype(int)

In [52]:

edges = edges.drop(columns=['properties', 'id'])

In [53]:
# prompt: create a unique id for each row, make sure it is of type int64
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
edges.head()
edges['id'].dtype

dtype('int64')

In [54]:

edges.keys()

Index(['type', 'start', 'end', 'label', 'weight', 'id'], dtype='object')

In [55]:
# prompt: show me the 1st record in the tier2_node_ids

type(tier2_node_ids[0])

int

In [56]:
# prompt: delete the last record in edges

# edges = edges[:-1]
# edges.tail()

In [57]:
# prompt: add the following record to edges:
# id=9999, properties={}, type="relationship", start=3, end=4,label="WriteDACL", weight=20

# import pandas as pd
# import numpy as np
# new_record = pd.DataFrame([{
#     'id': 9999,
#     'properties': {},
#     'type': 'relationship',
#     'start':  3,
#     'end': 4,
#     'label': 'WriteDacl',  # Corrected the typo to 'WriteDacl'
#     'weight': 20
# }])

# edges = pd.concat([edges, new_record], ignore_index=True)

# # Re-assign ids after adding a new record to ensure uniqueness and sequential order
# edges['id'] = np.arange(len(edges)).astype('int64')

# edges.tail()


In [58]:
tier2_node_ids[5]


18

In [59]:
## prompt: filter where edges['id'] = tier2_node_ids and call the dataframe selected_ddges

#selected_edges = edges[edges['start'].isin(tier2_node_ids)]

In [60]:
#edges['start'].head()
#edges.keys()

In [61]:
#selected_edges_t1 = edges[edges['start'].isin(tier1_node_ids)]

In [62]:
#selected_edges_t1

In [63]:
#selected_edges_t0 = edges[edges['start'].isin(tier0_node_ids)]
#selected_edges_t0

In [64]:
#tier0_node_ids

In [65]:
# prompt: show me type of columns in edges

edges.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3974 entries, 1297 to 5270
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   type    3974 non-null   object
 1   start   3974 non-null   object
 2   end     3974 non-null   object
 3   label   3974 non-null   object
 4   weight  3974 non-null   int64 
 5   id      3974 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 346.4+ KB


In [66]:
# prompt: make edges['start'] and edges['end'] int64

edges['start'] = edges['start'].astype('int64')
edges['end'] = edges['end'].astype('int64')

In [67]:
# describe = edges.describe()
# describe

In [68]:
print(edges[edges['start'] == 307])

              type  start  end       label  weight    id
2207  relationship    307   74    MemberOf       1   910
2757  relationship    307   59    MemberOf       1  1460
5034  relationship    307  104  HasSession       9  3737


In [69]:
# selected_edges_t2 = edges[edges['start'].isin(tier2_node_ids)]

In [70]:
# selected_edges


In [71]:
# selected_edges_t0 = edges[edges['start'].isin(tier0_node_ids)]
# selected_edges_t0

In [72]:
# selected_edges_t1 = edges[edges['start'].isin(tier1_node_ids)]
# selected_edges_t1

In [73]:
# tier0_node_ids

In [74]:
# selected_edges.head()

In [75]:
print(nodes[nodes['id'] == 8])

   id  \
8   8   

                                                                                                                                                                                                                                                                                                      properties  \
8  {'domain': 'MARK.LOCAL', 'name': 'T0 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-b9fd1aed-b3e6-4d53-9242-b85ff389822f', 'distinguishedname': 'OU=T0 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}   

   type    l1  l2    l3  weight  
8  node  Base  OU  None      20  


In [76]:
selected_nodes = nodes[nodes['id'].isin(tier2_node_ids)]

In [77]:
# prompt: set weight to 20 in selected_nodes

# for value in selected_nodes['id']:
#     selected_nodes.loc[selected_nodes['id'] == value, 'weight'] = 20


In [78]:
# selected_nodes

In [79]:
len(selected_nodes)

949

In [80]:
len(selected_edges_t0)

NameError: name 'selected_edges_t0' is not defined

In [ ]:
# import networkx as nx

# # Create a directed graph
# G = nx.DiGraph()

# # Add nodes with their weights
# for index, row in nodes.iterrows():
#     # Ensure node IDs are integers when adding to the graph
#     G.add_node(int(row['id']), weight=row['weight'])

# # Add edges with their weights
# for index, row in edges.iterrows():
#     # Ensure edge endpoints are integers when adding to the graph
#     G.add_edge(int(row['start']), int(row['end']), weight=row['weight'])

# # Compute shortest paths from each source node to all reachable nodes
# shortest_paths = {}
# for source in G.nodes():
#     # Use single_source_dijkstra_path to find paths from one source to all others
#     shortest_paths[source] = nx.single_source_dijkstra_path(G, source)

# # Filter out paths from a node to itself and store in a new dictionary
# shortest_paths_filtered = {
#     source: {
#         target: path
#         for target, path in paths.items()
#         if source != target
#     }
#     for source, paths in shortest_paths.items()
# }

# # Print the shortest paths (optional)
# # for source, paths in shortest_paths_filtered.items():
# #     print(f"Shortest paths from node {source}:")
# #     for target, path in paths.items():
# #         print(f"  To node {target}: {path}")

In [ ]:
# tier2_node_ids


In [ ]:
# tier0_node_ids

In [81]:
selected_edges = edges[edges['end'].isin(tier0_node_ids)]
len(selected_edges)

136

In [ ]:
# prompt: selected_edges=all edges excluding where edges['start'] is in tier0_node_ids

# selected_edges = edges[~edges['start'].isin(tier0_node_ids)]
# len(selected_edges)

In [ ]:
# selected_edges['end'] = selected_edges['end'].apply(lambda x: x['id']).astype(int)
# selected_edges['start'] = selected_edges['start'].apply(lambda x: x['id']).astype(int)

In [ ]:
#selected_edges_t2

In [82]:
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes with their weights
for index, row in selected_nodes.iterrows():
    # Ensure node IDs are integers when adding to the graph
    G.add_node(int(row['id']), weight=row['weight'])

# Add edges with their weights
for index, row in selected_edges.iterrows():
    # Ensure edge endpoints are integers when adding to the graph
    G.add_edge(row['start'], row['end'], weight=row['weight'])

# Compute shortest paths from each source node to all reachable nodes
shortest_paths = {}
for source in G.nodes():
    # Use single_source_dijkstra_path to find paths from one source to all others
    shortest_paths[source] = nx.single_source_dijkstra_path(G, source)

# Filter out paths from a node to itself and store in a new dictionary
shortest_paths_filtered = {
    source: {
        target: path
        for target, path in paths.items()
        if source != target
        if target in tier0_node_ids
    }
    for source, paths in shortest_paths.items()
}

# Print the shortest paths (optional)
for source, paths in shortest_paths_filtered.items():
  if paths:
    #print(f"Shortest paths from node {source}:")
    for target, path in paths.items():
        #print(f"  To node {target}: {path}")
        ShortestPath = source, target, path
        print(ShortestPath)

(15, 301, [15, 301])
(15, 318, [15, 318])
(26, 331, [26, 331])
(26, 341, [26, 341])
(26, 357, [26, 357])
(26, 405, [26, 405])
(26, 444, [26, 444])
(26, 448, [26, 448])
(27, 392, [27, 392])
(29, 481, [29, 481])
(486, 444, [486, 444])
(490, 331, [490, 331])
(494, 448, [494, 448])
(496, 444, [496, 444])
(498, 357, [498, 357])
(498, 448, [498, 448])
(499, 357, [499, 357])
(503, 444, [503, 444])
(505, 448, [505, 448])
(516, 405, [516, 405])
(517, 357, [517, 357])
(517, 448, [517, 448])
(524, 448, [524, 448])
(526, 405, [526, 405])
(538, 448, [538, 448])
(541, 331, [541, 331])
(542, 357, [542, 357])
(549, 357, [549, 357])
(552, 444, [552, 444])
(571, 341, [571, 341])
(572, 444, [572, 444])
(574, 331, [574, 331])
(581, 331, [581, 331])
(581, 444, [581, 444])
(583, 405, [583, 405])
(584, 448, [584, 448])
(587, 405, [587, 405])
(605, 448, [605, 448])
(608, 444, [608, 444])
(611, 331, [611, 331])
(611, 448, [611, 448])
(612, 405, [612, 405])
(616, 448, [616, 448])
(628, 448, [628, 448])
(629, 35

In [83]:
# prompt: using the above code, save all ShortestPath to a json file

shortest_paths_list = []
for source, paths in shortest_paths_filtered.items():
    if paths:
        for target, path in paths.items():
            shortest_paths_list.append({
                'source': source,
                'target': target,
                'path': path,
            })

with open('shortest_paths.json', 'w') as f:
    json.dump(shortest_paths_list, f, indent=4)

print("Shortest paths saved to shortest_paths.json")


Shortest paths saved to shortest_paths.json


In [84]:
# # prompt: save the df_compuers as json to content

# import json

# # Assuming df_computers is already defined as in your provided code.

# # Convert the DataFrame to a JSON string.
# selected_edges_json = selected_edges_t2.to_json(orient='records')

# # Save the JSON string to a file named 'content.json'.
# with open('selected_edges_t2.json', 'w') as f:
#   f.write(selected_edges_json)

In [ ]:
# prompt: save the df_compuers as json to content

# import json

# # Assuming df_computers is already defined as in your provided code.

# # Convert the DataFrame to a JSON string.
# selected_nodes_json = selected_nodes.to_json(orient='records')

# # Save the JSON string to a file named 'content.json'.
# with open('selected_nodes.json', 'w') as f:
#   f.write(selected_nodes_json)

In [ ]:
# prompt: export shortest_paths_filtered to json and save as shortest_paths_filtered_json

# shortest_paths_filtered_json = json.dumps(ShortestPath, indent=2)

# # Optionally, save the JSON string to a file
# with open('ShortestPath.json', 'w') as f:
#   f.write(shortest_paths_filtered_json)

In [85]:
# prompt: find the corresponding record in edges where (edges['start']=shortest_paths_list['source']) and (edges['end']=shortest_paths_list['target'] and then add up the weight

# Calculate the total weight for each shortest path
shortest_paths_with_weight = []
for path_info in shortest_paths_list:
    total_weight = 0
    # The path is a list of node IDs
    path = path_info['path']
    # Iterate through the path to get the edges
    for i in range(len(path) - 1):
        start_node = path[i]
        end_node = path[i+1]
        # Find the corresponding edge in the 'edges' DataFrame
        # There might be multiple edges between two nodes, so we sum up weights if needed
        # For shortest path, we typically consider the direct edge weight
        edge = edges[(edges['start'] == start_node) & (edges['end'] == end_node)]
        if not edge.empty:
            # Assuming there is only one edge with this start and end for simplicity in path finding
            # If multiple edges exist, you might need to adjust based on which edge is part of the shortest path calculation
            total_weight += edge.iloc[0]['weight']
    path_info['total_weight'] = total_weight
    shortest_paths_with_weight.append(path_info)

# Print or use the shortest_paths_with_weight list
for path_info in shortest_paths_with_weight:
    print(f"Path from {path_info['source']} to {path_info['target']}: {path_info['path']} - Total Weight: {path_info['total_weight']}")


Path from 15 to 301: [15, 301] - Total Weight: 2
Path from 15 to 318: [15, 318] - Total Weight: 2
Path from 26 to 331: [26, 331] - Total Weight: 2
Path from 26 to 341: [26, 341] - Total Weight: 2
Path from 26 to 357: [26, 357] - Total Weight: 2
Path from 26 to 405: [26, 405] - Total Weight: 2
Path from 26 to 444: [26, 444] - Total Weight: 2
Path from 26 to 448: [26, 448] - Total Weight: 2
Path from 27 to 392: [27, 392] - Total Weight: 2
Path from 29 to 481: [29, 481] - Total Weight: 2
Path from 486 to 444: [486, 444] - Total Weight: 9
Path from 490 to 331: [490, 331] - Total Weight: 9
Path from 494 to 448: [494, 448] - Total Weight: 9
Path from 496 to 444: [496, 444] - Total Weight: 9
Path from 498 to 357: [498, 357] - Total Weight: 9
Path from 498 to 448: [498, 448] - Total Weight: 9
Path from 499 to 357: [499, 357] - Total Weight: 9
Path from 503 to 444: [503, 444] - Total Weight: 9
Path from 505 to 448: [505, 448] - Total Weight: 9
Path from 516 to 405: [516, 405] - Total Weight: 9


In [86]:
# prompt: show me the top 10 weights from path_info['total_weight']

# Sort the paths by total weight in descending order
sorted_paths_by_weight = sorted(shortest_paths_with_weight, key=lambda x: x['total_weight'], reverse=True)

# Get the top 10 paths
top_10_paths = sorted_paths_by_weight[:10]

# Print the top 10 paths and their weights
print("\nTop 10 Paths by Total Weight:")
for path_info in top_10_paths:
    print(f"Path from {path_info['source']} to {path_info['target']}: {path_info['path']} - Total Weight: {path_info['total_weight']}")


Top 10 Paths by Total Weight:
Path from 60 to 303: [60, 8, 686, 303] - Total Weight: 20
Path from 60 to 296: [60, 8, 527, 296] - Total Weight: 20
Path from 60 to 305: [60, 8, 607, 305] - Total Weight: 20
Path from 60 to 457: [60, 8, 610, 457] - Total Weight: 20
Path from 60 to 324: [60, 8, 625, 324] - Total Weight: 20
Path from 60 to 307: [60, 8, 631, 307] - Total Weight: 20
Path from 60 to 692: [60, 8, 653, 692] - Total Weight: 20
Path from 60 to 458: [60, 8, 653, 458] - Total Weight: 20
Path from 869 to 303: [869, 8, 686, 303] - Total Weight: 13
Path from 869 to 296: [869, 8, 527, 296] - Total Weight: 13


In [87]:
# prompt:  show me the top 10 minimum  weights from path_info['total_weight']

# Sort the paths by total weight in ascending order
sorted_paths_by_weight_min = sorted(shortest_paths_with_weight, key=lambda x: x['total_weight'], reverse=False)

# Get the top 10 paths with minimum weights
top_10_min_paths = sorted_paths_by_weight_min[:10]

# Print the top 10 minimum paths and their weights
print("\nTop 10 Paths by Minimum Total Weight:")
for path_info in top_10_min_paths:
    print(f"Path from {path_info['source']} to {path_info['target']}: {path_info['path']} - Total Weight: {path_info['total_weight']}")


Top 10 Paths by Minimum Total Weight:
Path from 110 to 8: [110, 8] - Total Weight: 1
Path from 111 to 8: [111, 8] - Total Weight: 1
Path from 135 to 9: [135, 9] - Total Weight: 1
Path from 137 to 4: [137, 4] - Total Weight: 1
Path from 139 to 9: [139, 9] - Total Weight: 1
Path from 141 to 10: [141, 10] - Total Weight: 1
Path from 147 to 4: [147, 4] - Total Weight: 1
Path from 160 to 9: [160, 9] - Total Weight: 1
Path from 172 to 4: [172, 4] - Total Weight: 1
Path from 179 to 10: [179, 10] - Total Weight: 1


In [88]:
# prompt: show me the top 10 numbers that appears most in path_info['path']

from collections import Counter

# Extract all numbers from the 'path' lists
all_numbers_in_paths = [number for path_info in shortest_paths_list for number in path_info['path']]

# Count the occurrences of each number
number_counts = Counter(all_numbers_in_paths)

# Get the top 10 most common numbers and their counts
top_10_numbers = number_counts.most_common(10)

# Print the top 10 numbers and their counts
print("Top 10 numbers appearing most in path_info['path']:")
for number, count in top_10_numbers:
    print(f"Number: {number}, Count: {count}")

Top 10 numbers appearing most in path_info['path']:
Number: 8, Count: 287
Number: 4, Count: 185
Number: 7, Count: 119
Number: 10, Count: 56
Number: 1, Count: 31
Number: 137, Count: 31
Number: 147, Count: 31
Number: 172, Count: 31
Number: 208, Count: 31
Number: 653, Count: 29


In [120]:
# prompt: find paths from path_info['path'] that include the 1st entry in top_10_numbers with the highest and lowest weight

# Get the first number from the top 10 list
if top_10_numbers:
    target_number = top_10_numbers[1][0]
    print(f"\nSearching for paths containing the number: {target_number}")

    # Filter paths that include the target number
    paths_with_target = [
        path_info for path_info in shortest_paths_with_weight
        if target_number in path_info['path']
    ]

    if paths_with_target:
        print(f"\nPaths containing the number {target_number}:")
        print(nodes[nodes['id'] == target_number]['properties'].iloc[0]['distinguishedname'])

        # Sort the filtered paths by total weight in descending order
        paths_with_target_sorted_desc = sorted(paths_with_target, key=lambda x: x['total_weight'], reverse=True)

        # Get the path with the highest weight that includes the target number
        highest_weight_path_with_target = paths_with_target_sorted_desc[0]

        print(f"\nPath with the HIGHEST weight containing {target_number}:")
        print(f"Path from {highest_weight_path_with_target['source']} to {highest_weight_path_with_target['target']}: {highest_weight_path_with_target['path']} - Total Weight: {highest_weight_path_with_target['total_weight']}")

        # Sort the filtered paths by total weight in ascending order
        paths_with_target_sorted_asc = sorted(paths_with_target, key=lambda x: x['total_weight'], reverse=False)

        # Get the path with the lowest weight that includes the target number
        lowest_weight_path_with_target = paths_with_target_sorted_asc[0]

        print(f"\nPath with the LOWEST weight containing {target_number}:")
        print(f"Path from {lowest_weight_path_with_target['source']} to {lowest_weight_path_with_target['target']}: {lowest_weight_path_with_target['path']} - Total Weight: {lowest_weight_path_with_target['total_weight']}")


    else:
        print(f"No paths found containing the number {target_number}.")
else:
    print("The top_10_numbers list is empty.")


Searching for paths containing the number: 4

Paths containing the number 4:
OU=T0 ADMIN,DC=MARK,DC=LOCAL

Path with the HIGHEST weight containing 4:
Path from 1 to 692: [1, 4, 7, 692] - Total Weight: 6

Path with the LOWEST weight containing 4:
Path from 137 to 4: [137, 4] - Total Weight: 1


In [118]:
# prompt: for each value pair in highest_weight_path_with_target print out the corresponding edge record, eg if path contains 60,8,686,303, then pick out the 1st two values 60 and 8 and find the corresponding record in edges start and end

print("\nEdge records for the highest weight path containing the target number:")
path = highest_weight_path_with_target['path']
for i in range(len(path) - 1):
    start_node = path[i]
    end_node = path[i+1]
    # Find the corresponding edge record
    edge_record = edges[(edges['start'] == start_node) & (edges['end'] == end_node)]
    if not edge_record.empty:
        print(edge_record.iloc[0].to_dict())
        print(f"Start Node: ", start_node )
        print(nodes[nodes['id'] == start_node]['properties'].iloc[0]['distinguishedname'])
        print(f"End Node: ", end_node )
        print(nodes[nodes['id'] == end_node]['properties'].iloc[0]['distinguishedname'])
        print("-------------------------------------------")
    else:
        print(f"No edge found between nodes {start_node} and {end_node}")



Edge records for the highest weight path containing the target number:
{'type': 'relationship', 'start': 1, 'end': 4, 'label': 'Contains', 'weight': 2, 'id': 3}
Start Node:  1
OU=ADMIN,DC=MARK,DC=LOCAL
End Node:  4
OU=T0 ADMIN,DC=MARK,DC=LOCAL
-------------------------------------------
{'type': 'relationship', 'start': 4, 'end': 7, 'label': 'Contains', 'weight': 2, 'id': 6}
Start Node:  4
OU=T0 ADMIN,DC=MARK,DC=LOCAL
End Node:  7
OU=T0 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL
-------------------------------------------
{'type': 'relationship', 'start': 7, 'end': 692, 'label': 'Contains', 'weight': 2, 'id': 1425}
Start Node:  7
OU=T0 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL
End Node:  692
CN=T0 DEFAULT ADMIN USER,OU=T0 ADMIN ACCOUNTS,OU=T0 Admin,OU=Admin,DC=MARK,DC=LOCAL
-------------------------------------------


In [117]:
# prompt: how would i print out nodes['distinguishedname'] and nodes['start] when nodes[nodes['id'] == start_node]

# Access the row where the 'id' column matches start_node
node_row = nodes[nodes['id'] == start_node]

# Check if a matching row was found
if not node_row.empty:
    # Access the 'properties' dictionary within that row and then the 'distinguishedname' key
    distinguished_name = node_row['properties'].iloc[0].get('distinguishedname')
    start_value = node_row['start'].iloc[0] # Access the 'start' column directly

    # Print the values
    print(f"Distinguished Name: {distinguished_name}")
    print(f"Start: {start_value}")
else:
    print(f"Node with ID {start_node} not found.")



KeyError: 'start'

In [111]:

print("\nEdge records for the lowest weight path containing the target number:")
path = lowest_weight_path_with_target['path']
for i in range(len(path) - 1):
    start_node = path[i]
    end_node = path[i+1]
    # Find the corresponding edge record
    edge_record = edges[(edges['start'] == start_node) & (edges['end'] == end_node)]
    if not edge_record.empty:
        print(edge_record.iloc[0].to_dict())
        print(nodes[nodes['id'] == start_node])
        print(nodes[nodes['id'] == end_node])
    else:
        print(f"No edge found between nodes {start_node} and {end_node}")



Edge records for the lowest weight path containing the target number:
{'type': 'relationship', 'start': 137, 'end': 4, 'label': 'GpLink', 'weight': 1, 'id': 139}
      id  \
137  137   

                                                                                                                                                                                                  properties  \
137  {'domain': 'MARK.LOCAL', 'name': 'Network_Restriction_23@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-366cbb81-f13b-490d-9207-67a0ff5d207d', 'exploitable': False, 'owned': False}   

     type    l1   l2    l3  weight  
137  node  Base  GPO  None       1  
   id  \
4   4   

                                                                                                                                                                                                                                                                                      properties  \
4  {'domai

In [95]:
print(nodes[nodes['id'] == 110])

      id  \
110  110   

                                                                                                                                                                       properties  \
110  {'domain': 'MARK.LOCAL', 'name': 'PAW Configuration@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-d37c9c32-7ad3-4495-862c-b8c9734a1860', 'owned': False}   

     type    l1   l2    l3  weight  
110  node  Base  GPO  None       1  


In [ ]:
# prompt: convert shortest_paths_with_weight to dataframe called df_shortest_paths_with_weight

import pandas as pd
df_shortest_paths_with_weight = pd.DataFrame(shortest_paths_with_weight)

In [ ]:
df_shortest_paths_with_weight

,source,target,path,weight,total_weight
0,3,481,"[3, 23, 29, 481]",NaN,6
1,3,331,"[3, 24, 26, 331]",NaN,6
2,3,341,"[3, 24, 26, 341]",NaN,6
3,3,357,"[3, 24, 26, 357]",NaN,6
4,3,405,"[3, 24, 26, 405]",NaN,6
...,...,...,...,...,...
979,873,405,"[873, 28, 516, 405]",NaN,19
980,873,341,"[873, 28, 571, 341]",NaN,19
981,873,331,"[873, 28, 581, 331]",NaN,19
982,873,8,"[873, 28, 492, 386, 705, 97, 711, 869, 8]",NaN,30


In [ ]:
# prompt: for each source find minimum weight and target. if weight is the same print out all instances

import pandas as pd
# Group by source and find the minimum weight
min_weights = df_shortest_paths_with_weight.groupby('source')['total_weight'].min().reset_index()

# Merge with the original DataFrame to get rows with the minimum weight
result_min = pd.merge(df_shortest_paths_with_weight, min_weights, on=['source', 'total_weight'])

# Print the results
# for index, row in result.iterrows():
#   print(f"Source: {row['source']}, Target: {row['target']}, Minimum Weight: {row['total_weight']}, Path: {row['path']}")


In [ ]:
# prompt: for each source find minimum weight and target. if weight is the same print out all instances

import pandas as pd
# Group by source and find the minimum weight
max_weights = df_shortest_paths_with_weight.groupby('source')['total_weight'].max().reset_index()

# Merge with the original DataFrame to get rows with the minimum weight
result_max = pd.merge(df_shortest_paths_with_weight, max_weights, on=['source', 'total_weight'])

# Print the results
# for index, row in result.iterrows():
#   print(f"Source: {row['source']}, Target: {row['target']}, Max Weight: {row['total_weight']}, Path: {row['path']}")
# df_shortest_paths_with_weight = pd.DataFrame(shortest_paths_with_weight)
# df_shortest_paths_with_weight

In [ ]:
result_min

,source,target,path,weight,total_weight
0,3,481,"[3, 23, 29, 481]",NaN,6
1,3,331,"[3, 24, 26, 331]",NaN,6
2,3,341,"[3, 24, 26, 341]",NaN,6
3,3,357,"[3, 24, 26, 357]",NaN,6
4,3,405,"[3, 24, 26, 405]",NaN,6
...,...,...,...,...,...
449,873,357,"[873, 28, 498, 357]",NaN,19
450,873,448,"[873, 28, 498, 448]",NaN,19
451,873,405,"[873, 28, 516, 405]",NaN,19
452,873,341,"[873, 28, 571, 341]",NaN,19


In [ ]:
result_max

,source,target,path,weight,total_weight
0,3,8,"[3, 22, 32, 711, 869, 8]",NaN,10
1,6,444,"[6, 17, 97, 26, 444]",NaN,13
2,6,448,"[6, 17, 97, 26, 448]",NaN,13
3,6,357,"[6, 17, 97, 26, 357]",NaN,13
4,6,405,"[6, 17, 97, 26, 405]",NaN,13
...,...,...,...,...,...
250,870,8,"[870, 30, 495, 377, 705, 97, 711, 869, 8]",NaN,30
251,871,481,"[871, 30, 495, 377, 705, 97, 711, 869, 29, 481]",NaN,38
252,872,341,"[872, 30, 495, 377, 705, 97, 26, 341]",NaN,30
253,872,8,"[872, 30, 495, 377, 705, 97, 711, 869, 8]",NaN,30


In [ ]:
# prompt: show me where source appears the most within the results_max dataframe. is ther one source that appears most

source_counts = result_max['source'].value_counts()

print("Count of each source in result_max:")
print(source_counts)

most_frequent_source = source_counts.index[0]
most_frequent_count = source_counts.iloc[0]

print(f"\nThe source that appears most frequently is: {most_frequent_source} with a count of {most_frequent_count}")

if most_frequent_count > 1:
  print("Yes, there is one source that appears most.")
else:
  print("No single source appears most frequently (or all sources appear only once).")

Count of each source in result_max:
source
6      7
22     7
16     7
321    7
313    7
      ..
705    1
868    1
870    1
871    1
873    1
Name: count, Length: 137, dtype: int64

The source that appears most frequently is: 6 with a count of 7
Yes, there is one source that appears most.


In [ ]:
# prompt: convert result into a DataFrame

import pandas as pd
df_shortest_paths_with_weight = pd.DataFrame(shortest_paths_with_weight)
df_shortest_paths_with_weight

,source,target,path,weight,total_weight
0,3,481,"[3, 23, 29, 481]",NaN,6
1,3,331,"[3, 24, 26, 331]",NaN,6
2,3,341,"[3, 24, 26, 341]",NaN,6
3,3,357,"[3, 24, 26, 357]",NaN,6
4,3,405,"[3, 24, 26, 405]",NaN,6
...,...,...,...,...,...
979,873,405,"[873, 28, 516, 405]",NaN,19
980,873,341,"[873, 28, 571, 341]",NaN,19
981,873,331,"[873, 28, 581, 331]",NaN,19
982,873,8,"[873, 28, 492, 386, 705, 97, 711, 869, 8]",NaN,30


In [ ]:
print(edges[edges['end'] == 301])

        id properties          type  start  end       label  weight
1448  1448         {}  relationship     15  301    Contains       2
3099  3099         {}  relationship    637  301  HasSession       9


In [ ]:
shortest_paths_with_weight

[{'source': 3,
  'target': 481,
  'path': [3, 23, 29, 481],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 331,
  'path': [3, 24, 26, 331],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 341,
  'path': [3, 24, 26, 341],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 357,
  'path': [3, 24, 26, 357],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 405,
  'path': [3, 24, 26, 405],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 444,
  'path': [3, 24, 26, 444],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 448,
  'path': [3, 24, 26, 448],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 392,
  'path': [3, 24, 27, 392],
  'weight': np.float64(nan),
  'total_weight': np.int64(6)},
 {'source': 3,
  'target': 8,
  

In [ ]:
# prompt: for each node in shortest_paths_with_weight, find the minimum total_weight and the maxium total_weight, save it to a DataFrame

import pandas as pd
# Initialize a dictionary to store min and max weights for each target node
target_node_weights = {}

# Iterate through the shortest paths
for path_info in shortest_paths_with_weight:
    target_node = path_info['target']
    total_weight = path_info['total_weight']

    if target_node not in target_node_weights:
        target_node_weights[target_node] = {'min_weight': float('inf'), 'max_weight': float('-inf')}

    # Update min and max weights
    target_node_weights[target_node]['min_weight'] = min(target_node_weights[target_node]['min_weight'], total_weight)
    target_node_weights[target_node]['max_weight'] = max(target_node_weights[target_node]['max_weight'], total_weight)

# Convert the dictionary to a DataFrame
min_max_weights_df = pd.DataFrame.from_dict(target_node_weights, orient='index')
min_max_weights_df.index.name = 'target_node'
min_max_weights_df = min_max_weights_df.reset_index()

# Display the resulting DataFrame
print("\nMin and Max Total Weights for Each Target Node:")
min_max_weights_df


Min and Max Total Weights for Each Target Node:


,target_node,min_weight,max_weight
0,481,2,38
1,331,2,39
2,341,2,39
3,357,2,39
4,405,2,39
5,444,2,39
6,448,2,39
7,392,2,34
8,8,2,39
9,301,2,29


In [ ]:
# prompt: for each record in "shortest_paths_filtered" add up the weights and find the smallest and largest combined weights and the path for the largest weight

# Function to calculate the total weight of a path
def calculate_path_weight(graph, path):
    total_weight = 0
    for i in range(len(path) - 1):
        # Add node weight
        total_weight += graph.nodes[path[i]].get('weight', 0)
        # Add edge weight
        total_weight += graph[path[i]][path[i+1]].get('weight', 0)
    # Add the weight of the last node in the path
    if path:
        total_weight += graph.nodes[path[-1]].get('weight', 0)
    return total_weight

all_path_weights = {}

# Iterate through each source and its paths in shortest_paths_filtered
for source, paths_from_source in shortest_paths_list.items():
    all_path_weights[source] = {}
    # Iterate through each target and its path
    for target, path in paths_from_source.items():
        # Calculate the weight of the path using the original graph G
        weight = calculate_path_weight(G, path)
        all_path_weights[source][target] = weight

# Find the smallest and largest combined weights
min_weight = float('inf')
max_weight = float('-inf')
largest_weight_path = None
largest_weight_source = None
largest_weight_target = None

for source, paths_from_source in all_path_weights.items():
    for target, weight in paths_from_source.items():
        if weight < min_weight:
            min_weight_source = source
            min_weight_target = target
            min_weight_path = shortest_paths_filtered[source][target]
            min_weight = weight
        if weight > max_weight:
            max_weight = weight
            largest_weight_source = source
            largest_weight_target = target
            largest_weight_path = shortest_paths_filtered[source][target]

print(f"Smallest combined weight: {min_weight}")
print(f"Path for the smallest weight: {min_weight_path}")
print(f"Source for the smallest weight path: {min_weight_source}")
print(f"Target for the smallest weight path: {min_weight_target}")
print(f"Largest combined weight: {max_weight}")
print(f"Path for the largest weight: {largest_weight_path}")
print(f"Source for the largest weight path: {largest_weight_source}")
print(f"Target for the largest weight path: {largest_weight_target}")

AttributeError: 'list' object has no attribute 'items'

In [ ]:

print(nodes[nodes['id'] == 637])
print(nodes[nodes['id'] == 301])
# print(nodes[nodes['id'] == 94])
# print(nodes[nodes['id'] == 698])
# print(nodes[nodes['id'] == 710])
# print(nodes[nodes['id'] == 91])

      id                                         properties  type    l1  \
637  637  {'name': 'PAW-00018@MARK.LOCAL', 'operatingsys...  node  Base   

           l2    l3  weight  
637  Computer  None    20.0  
      id                                         properties  type    l1    l2  \
301  301  {'domain': 'MARK.LOCAL', 'objectid': 'S-1-5-21...  node  Base  User   

       l3  weight  
301  None    20.0  


In [ ]:
selected_nodes

,id,properties,type,l1,l2,l3,weight
3,3,"{'domain': 'MARK.LOCAL', 'name': 'Tier 2@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5de20ac8-e7ff-4f05-b597-268af392400e', 'distinguishedname': 'OU=TIER 2,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
6,6,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e4602f9d-12bf-4f9e-b667-f6ccd27fae90', 'distinguishedname': 'OU=T2 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
15,15,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-61c17d81-4eb2-4829-94e7-c7ddd6ecd193', 'distinguishedname': 'OU=T2 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
16,16,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-5305b84a-5f72-4ee8-9726-81485bc11905', 'distinguishedname': 'OU=T2 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
17,17,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-508809b7-63d4-4f53-8376-b31aa66c0af6', 'distinguishedname': 'OU=T2 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
18,18,"{'domain': 'MARK.LOCAL', 'name': 'T2 Admin Service Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-e7679546-c753-4556-889a-67fd84f9d165', 'distinguishedname': 'OU=T2 ADMIN SERVICE ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
22,22,"{'domain': 'MARK.LOCAL', 'name': 'T2 Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f6d13355-d2f9-45cf-9ee6-fdc8230ae4e5', 'distinguishedname': 'OU=T2 GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
23,23,"{'domain': 'MARK.LOCAL', 'name': 'T2 Workstations@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-66aadca5-ccbc-46d4-91e7-f74f2527bdb5', 'distinguishedname': 'OU=T2 WORKSTATIONS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
24,24,"{'domain': 'MARK.LOCAL', 'name': 'T2 User Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-d4c0667d-16e8-4afd-8cf3-5f6a900f6913', 'distinguishedname': 'OU=T2 USER ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1
25,25,"{'domain': 'MARK.LOCAL', 'name': 'T2 Servers@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-3f1588f2-5124-4111-8d63-caf4b57a87ca', 'distinguishedname': 'OU=T2 SERVERS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,1


In [ ]:
selected_nodes

,id,properties,type,l1,l2,l3,weight
0,0,"{'domain': 'MARK.LOCAL', 'name': 'MARK.LOCAL', 'highvalue': True, 'objectid': 'S-1-5-21-883232822-274137685-4173207997', 'distinguishedname': 'DC=MARK,DC=LOCAL', 'functionallevel': '2012', 'owned': False}",node,Base,Domain,None,20
4,4,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-f249f685-d5e0-4dad-b5b5-7f723c41ca1e', 'distinguishedname': 'OU=T0 ADMIN,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
7,7,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-c0fb853d-9b96-449b-b25c-4d20a1c02e5f', 'distinguishedname': 'OU=T0 ADMIN ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
8,8,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Devices@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-b9fd1aed-b3e6-4d53-9242-b85ff389822f', 'distinguishedname': 'OU=T0 ADMIN DEVICES,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
9,9,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Groups@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-cd13b9d6-76ca-4b5c-82bd-2d86bef8a375', 'distinguishedname': 'OU=T0 ADMIN GROUPS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
10,10,"{'domain': 'MARK.LOCAL', 'name': 'T0 Admin Service Accounts@MARK.LOCAL', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-58cd5b4f-82db-4cd1-a3f4-08661c9c53d2', 'distinguishedname': 'OU=T0 ADMIN SERVICE ACCOUNTS,DC=MARK,DC=LOCAL', 'description': None, 'highvalue': False, 'blocksInheritance': False, 'owned': False}",node,Base,OU,None,20
36,36,"{'domain': 'MARK.LOCAL', 'name': 'ADMINISTRATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-544', 'highvalue': True, 'distinguishedname': 'CN=Administrators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Administrators have complete and unrestricted access to the computer/domain', 'admincount': True, 'owned': False}",node,Base,Group,None,20
38,38,"{'domain': 'MARK.LOCAL', 'name': 'PRINT OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-550', 'highvalue': True, 'distinguishedname': 'CN=Print Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer printers installed on domain controllers', 'admincount': True, 'owned': False}",node,Base,Group,None,20
40,40,"{'domain': 'MARK.LOCAL', 'name': 'BACKUP OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-551', 'highvalue': True, 'distinguishedname': 'CN=Backup Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Backup Operators can override security restrictions for the sole purpose of backing up or restoring files', 'admincount': True, 'owned': False}",node,Base,Group,None,20
59,59,"{'domain': 'MARK.LOCAL', 'name': 'SERVER OPERATORS@MARK.LOCAL', 'objectid': 'MARK.LOCAL-S-1-5-32-549', 'highvalue': True, 'distinguishedname': 'CN=Server Operators,CN=Builtin,DC=MARK,DC=LOCAL', 'description': 'Members can administer domain servers', 'admincount': True, 'owned': False}",node,Base,Group,None,20


In [ ]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
shortest_paths

{0: {0: [0],
  285: [0, 285],
  287: [0, 287],
  49: [0, 285, 49],
  74: [0, 285, 74],
  63: [0, 287, 63],
  86: [0, 285, 74, 86],
  87: [0, 285, 74, 87],
  50: [0, 285, 74, 87, 50]},
 4: {4: [4], 7: [4, 7]},
 7: {7: [7]},
 8: {8: [8],
  686: [8, 686],
  527: [8, 527],
  530: [8, 530],
  531: [8, 531],
  532: [8, 532],
  553: [8, 553],
  607: [8, 607],
  610: [8, 610],
  625: [8, 625],
  631: [8, 631],
  632: [8, 632],
  640: [8, 640],
  645: [8, 645],
  653: [8, 653],
  663: [8, 663]},
 9: {9: [9],
  65: [9, 65],
  284: [9, 65, 36, 284],
  285: [9, 65, 36, 285],
  286: [9, 65, 286],
  287: [9, 65, 36, 287],
  63: [9, 65, 64, 63],
  36: [9, 65, 36],
  37: [9, 65, 37],
  38: [9, 65, 286, 38],
  39: [9, 65, 39],
  40: [9, 65, 40],
  41: [9, 65, 36, 41],
  42: [9, 65, 36, 42],
  43: [9, 65, 43],
  45: [9, 65, 45],
  48: [9, 65, 36, 48],
  50: [9, 65, 36, 50],
  51: [9, 65, 36, 51],
  52: [9, 65, 52],
  53: [9, 65, 36, 53],
  56: [9, 65, 36, 56],
  57: [9, 65, 36, 57],
  58: [9, 65, 36, 58

In [ ]:
shortest_paths_filtered

{0: {285: [0, 285],
  287: [0, 287],
  49: [0, 285, 49],
  74: [0, 285, 74],
  63: [0, 287, 63],
  86: [0, 285, 74, 86],
  87: [0, 285, 74, 87],
  50: [0, 285, 74, 87, 50]},
 4: {7: [4, 7]},
 7: {},
 8: {686: [8, 686],
  527: [8, 527],
  530: [8, 530],
  531: [8, 531],
  532: [8, 532],
  553: [8, 553],
  607: [8, 607],
  610: [8, 610],
  625: [8, 625],
  631: [8, 631],
  632: [8, 632],
  640: [8, 640],
  645: [8, 645],
  653: [8, 653],
  663: [8, 663]},
 9: {65: [9, 65],
  284: [9, 65, 36, 284],
  285: [9, 65, 36, 285],
  286: [9, 65, 286],
  287: [9, 65, 36, 287],
  63: [9, 65, 64, 63],
  36: [9, 65, 36],
  37: [9, 65, 37],
  38: [9, 65, 286, 38],
  39: [9, 65, 39],
  40: [9, 65, 40],
  41: [9, 65, 36, 41],
  42: [9, 65, 36, 42],
  43: [9, 65, 43],
  45: [9, 65, 45],
  48: [9, 65, 36, 48],
  50: [9, 65, 36, 50],
  51: [9, 65, 36, 51],
  52: [9, 65, 52],
  53: [9, 65, 36, 53],
  56: [9, 65, 36, 56],
  57: [9, 65, 36, 57],
  58: [9, 65, 36, 58],
  59: [9, 65, 286, 59],
  60: [9, 65, 60]

In [ ]:
# prompt: can you display the combined weight from the shortest_paths and display the path with the max weight

# Function to calculate the total weight of a path
def calculate_path_weight(graph, path):
    total_weight = 0
    # Add the weight of the starting node
    if path:
        total_weight += graph.nodes[path[0]].get('weight', 0) # Get node weight
    # Add the weight of the edges and the weights of subsequent nodes
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        # Add edge weight
        total_weight += graph.edges[u, v].get('weight', 0) # Get edge weight
        # Add weight of the next node
        total_weight += graph.nodes[v].get('weight', 0) # Get node weight
    return total_weight

max_weight = -1
max_path = None

# Iterate through the filtered shortest paths and calculate total weights
for source, paths in shortest_paths_filtered.items():
    for target, path in paths.items():
        current_weight = calculate_path_weight(G, path)
#        print(f"Path from {source} to {target}: {path}, Combined Weight: {current_weight}")
        if current_weight > max_weight:
            max_weight = current_weight
            max_path = path

print(f"\nPath with the maximum combined weight: {max_path}")
print(f"Maximum combined weight: {max_weight}")


Path with the maximum combined weight: [9, 65, 36, 0]
Maximum combined weight: 87.0


In [ ]:
max_path
max_path = pd.DataFrame(max_path)
max_path.head()


,0
0,9
1,65
2,36
3,0


In [ ]:
rename = {0: 'id'}
max_path = max_path.rename(columns=rename)
max_path

,id
0,9
1,65
2,36
3,0


In [ ]:
# prompt: get label from nodes where id=max_path['id']

# Assuming max_path is a DataFrame with an 'id' column representing the node IDs in the max path
# and nodes is a DataFrame with 'id' and 'properties' columns.

# Get the list of node IDs from the max_path DataFrame
max_path_node_ids = max_path['id'].tolist()

# Filter the nodes DataFrame to get only the nodes that are in the max_path
nodes_in_max_path = nodes[nodes['id'].isin(max_path_node_ids)].copy()

# Now, extract the 'name' property from the 'properties' column for these nodes
# Use .loc to avoid SettingWithCopyWarning
# We use a lambda function with .apply to safely access nested dictionary keys.
max_path_node_labels = nodes_in_max_path.loc[:, 'properties'].apply(lambda x: x.get('name', ''))

print("Labels of nodes in the max path:")
print(max_path_node_labels.tolist())


Labels of nodes in the max path:
['MARK.LOCAL', 'T0 Admin Groups@MARK.LOCAL', 'ADMINISTRATORS@MARK.LOCAL', 'ENTERPRISE ADMINS@MARK.LOCAL']


In [ ]:
max_path

,id
0,9
1,65
2,36
3,0


ignore from here


In [ ]:
vertices = nodes['id'].tolist()
vertices


[6221,
 6222,
 6223,
 6224,
 6225,
 6226,
 6227,
 6228,
 6229,
 6230,
 6231,
 6232,
 6233,
 6234,
 6235,
 6236,
 6237,
 6238,
 6239,
 6240,
 6241,
 6242,
 6243,
 6244,
 6245,
 6246,
 6247,
 6248,
 6249,
 6250,
 6251,
 6252,
 6253,
 6254,
 6255,
 6256,
 6257,
 6258,
 6259,
 6260,
 6261,
 6262,
 6263,
 6264,
 6265,
 6266,
 6267,
 6268,
 6269,
 6270,
 6271,
 6272,
 6273,
 6274,
 6275,
 6276,
 6277,
 6278,
 6279,
 6280,
 6281,
 6282,
 6283,
 6284,
 6285,
 6286,
 6287,
 6288,
 6289,
 6290,
 6291,
 6292,
 6293,
 6294,
 6295,
 6296,
 6297,
 6298,
 6299,
 6300,
 6301,
 6302,
 6303,
 6304,
 6305,
 6306,
 6307,
 6308,
 6309,
 6310,
 6311,
 6312,
 6313,
 6314,
 6315,
 6316,
 6317,
 6318,
 6319,
 6320,
 6321,
 6322,
 6323,
 6324,
 6325,
 6326,
 6327,
 6328,
 6329,
 6330,
 6331,
 6332,
 6333,
 6334,
 6335,
 6336,
 6337,
 6338,
 6339,
 6340,
 6341,
 6342,
 6343,
 6344,
 6345,
 6346,
 6347,
 6348,
 6349,
 6350,
 6351,
 6352,
 6353,
 6354,
 6355,
 6356,
 6357,
 6358,
 6359,
 6360,
 6361,
 6362,
 6363,

In [ ]:
adj_list=adj.values.tolist()
len(adj_list)

48321

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def createAdjacencyMatrix(vertices,edges):
  noofvertices=len(vertices)
  adjM=[]
  while(len(adjM)<noofvertices):
    temp=[]
    for i in range(noofvertices):
      temp.append(0)
    adjM.append(temp)
  for edge in edges:
    i=edge[0]
    j=edge[1]
    if i>=noofvertices or j>=noofvertices or i<0 or j<0:
      print(f"Not a Proper Input in Edge {i},{j}")
    else:
      adjM[i][j]=1
      adjM[j][i]=1
#  G=nx.Graph()
#  G.add_edges_from(edges)
#  nx.draw_networkx(G)
#  plt.show()
  return adjM
vertices = vertices
edges = adj_list
adjM=createAdjacencyMatrix(vertices,edges)


Not a Proper Input in Edge 10075,6298
Not a Proper Input in Edge 10077,6298
Not a Proper Input in Edge 10078,6298
Not a Proper Input in Edge 10080,6298
Not a Proper Input in Edge 10081,6298
Not a Proper Input in Edge 10082,6298
Not a Proper Input in Edge 10083,6298
Not a Proper Input in Edge 10085,6298
Not a Proper Input in Edge 10087,6298
Not a Proper Input in Edge 10088,6298
Not a Proper Input in Edge 10089,6298
Not a Proper Input in Edge 10090,6298
Not a Proper Input in Edge 10094,6298
Not a Proper Input in Edge 10100,6298
Not a Proper Input in Edge 10101,6298
Not a Proper Input in Edge 10103,6298
Not a Proper Input in Edge 10104,6298
Not a Proper Input in Edge 10105,6298
Not a Proper Input in Edge 10106,6298
Not a Proper Input in Edge 10107,6298
Not a Proper Input in Edge 10108,6298
Not a Proper Input in Edge 10110,6298
Not a Proper Input in Edge 10111,6298
Not a Proper Input in Edge 10114,6298
Not a Proper Input in Edge 10116,6298
Not a Proper Input in Edge 10117,6298
Not a Proper

In [ ]:
#adjM.shape
import numpy as np
len(adjM)
arr = np.array(adjM)

newarr = arr.reshape(10075, 10075)

print(newarr)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
def create_adjacency_matrix(V, edges):
    # Initialize an empty V x V matrix with all zeros
    matrix = [[0] * V for _ in range(V)]

    # Populate the matrix based on the edges
    for edge in edges:
        u, v = edge
        matrix[u][v] = 1
#       matrix[v][u] = 1  # Undirected graph

    return matrix
V1 = len(adj)
edges1 = adj[['start', 'end']].values.tolist()
adj_matrix1 = create_adjacency_matrix(V1, edges1)
for row in adj_matrix1:
    print(row)
print()

In [ ]:
type(adj)

pandas.core.frame.DataFrame